# How to Add Custom Datasets to AtomDB
This notebook walks through the steps required to integrate a custom dataset into AtomDB.

## 1. Import Required Libraries


In [10]:
import os 
import numpy as np
import atomdb

## 2. Define Custom Dataset Structure

A custom dataset must live under `atomdb/datasets/<your_name>/` and include:

- A `run.py` file exposing a `run(elem, charge, mult, nexc, dataset, datapath)` function that compiles raw data into a `Species` instance.
- A `raw/` folder containing the raw data files that `run()` will read. This is where AtomDB looks for the source data (e.g. `.slater` files, Gaussian output files, etc.) before compiling them into the database cache.

```
atomdb/datasets/custom_ds/
├── run.py
└── raw/
    ├── H_0_1.mydata
    ├── He_0_1.mydata
    └── ...
```

## 3. Implement the `run()` Function

The `run()` function is the entry point AtomDB calls when compiling a species. It must read the raw data files from the `raw/` folder, compute the relevant properties, and return an `atomdb.Species` instance.

There are two existing datasets that serve as good reference implementations:

- **Slater** ([`atomdb/datasets/slater/run.py`](https://github.com/theochem/AtomDB/blob/master/atomdb/datasets/slater/run.py)): reads `.slater` files containing Hartree-Fock Slater-type orbital coefficients and exponents. A good template if your raw data describes atomic wavefunctions in terms of Slater-type orbitals.

- **Gaussian** ([`atomdb/datasets/gaussian/run.py`](https://github.com/theochem/AtomDB/blob/master/atomdb/datasets/gaussian/run.py)): reads Gaussian basis set output files. A good template if your raw data comes from Gaussian calculations using a contracted Gaussian basis set.

The key things `run()` must do:
1. Parse the raw data file for the given element, charge, and multiplicity.
2. Build a `fields` dictionary with the required `Species` fields (elem, atnum, nelec, nspin, etc.).
3. Return `atomdb.Species(dataset, fields)`.

Note that the return value is **not a plain dictionary** — it must be an `atomdb.Species` instance constructed from the fields dict.

In [ ]:

#run.py template
import atomdb
import numpy as np
from atomdb.periodic import Element
import os


def run(elem, charge, mult, nexc, dataset, datapath):
    """Compile a species from custom raw data into AtomDB."""
    #element info
    elem = atomdb.element_symbol(elem)
    atnum = atomdb.element_number(elem)
    nelec = atnum - charge
    nspin = mult - 1

    # Load your raw data file from the raw/ folder
    raw_file = os.path.join(datapath, dataset, "raw", f"{elem}_{charge}_{mult}.mydata")
    data = np.loadtxt(raw_file)
    rs       = data[:, 0] 
    dens_tot = data[:, 1] 
    
    # Get element-level properties (radii, mass, etc.) for neutral species
    atom = Element(elem)
    cov_radius, vdw_radius, at_radius, polarizability, dispersion = [None] * 5
    if charge == 0:
        cov_radius = atom.cov_radius
        vdw_radius = atom.vdw_radius
        at_radius = atom.at_radius
        polarizability = atom.pold
        dispersion = {"C6": atom.c6}

    # Build the fields dict  keys must match the Species constructor
    fields = dict(
        elem=elem,
        atnum=atnum,
        obasis_name=dataset, #name of your dataset
        nelec=nelec,
        nspin=nspin,
        nexc=nexc,
        atmass=atom.mass,
        cov_radius=cov_radius,
        vdw_radius=vdw_radius,
        at_radius=at_radius,
        polarizability=polarizability,
        dispersion=dispersion,
        energy=None,       # fill in from your raw data
        # ... add density, ked, mo arrays etc. as needed
    )

    # Must return a Species instance, not a plain dict
    return atomdb.Species(dataset, fields)

## 4. Compile and Store the Dataset

Once `run.py` and the `raw/` folder are in place, compile the raw data into 
AtomDB's binary cache using `atomdb.compile()`. This calls your `run()` function 
and writes the result to disk.
```python
atomdb.compile("H", charge=0, mult=2, nexc=0, dataset="custom_ds", datapath=datapath)
```
```bash
# CLI equivalent
atomdb compile custom_ds H 0 2
```


**Note on dependencies:** `compile()` imports and runs your dataset's `run.py` directly, 
so any packages your `run.py` depends on must be installed before calling it. 
For example, the built-in Slater dataset requires the `grid` package from the theochem 
group. Make sure you document the dependencies of your custom dataset similarly so 
other contributors know what to install.

For built-in datasets like Slater, pre-compiled data is downloaded automatically 
when you call `atomdb.load()` — you do not need to run `compile()` manually 
unless you are modifying the dataset itself.

## 5. Load and Validate the Custom Dataset

After compiling and dumping, use `atomdb.load()` to retrieve the species from the cache and confirm it was stored correctly.

In [ ]:
import atomdb

#Replace slater with your custom dataset
sp = atomdb.load("H", charge=0, mult=2, nexc=0, dataset="slater")

# Inspect basic species properties
print("Element:        ", sp.elem)
print("Atomic number:  ", sp.atnum)
print("Num electrons:  ", sp.nelec)
print("Charge:         ", sp.charge)
print("Multiplicity:   ", sp.mult)
print("Dataset:        ", sp.dataset)
print("Energy:         ", sp.energy)

Element:         H
Atomic number:   1
Num electrons:   1
Charge:          0
Multiplicity:    2
Dataset:         slater
Energy:          -0.5


## 6. Example: Using the Custom Dataset

Lets run through a full example on how to make a dataset

In [6]:
import os
import atomdb

# This is where we need to put our run.py
atomdb_path = os.path.dirname(atomdb.__file__)
datapath = os.path.join(atomdb_path, "datasets") 
dataset_path = os.path.join(atomdb_path, "datasets", "mydata")
raw_path = os.path.join(dataset_path, "raw")

os.makedirs(dataset_path, exist_ok=True)
os.makedirs(raw_path, exist_ok=True)

print("Dataset folder:", dataset_path)
print("Raw folder:    ", raw_path)

Dataset folder: /home/username/.local/lib/python3.13/site-packages/atomdb/datasets/mydata
Raw folder:     /home/username/.local/lib/python3.13/site-packages/atomdb/datasets/mydata/raw


In [7]:
import numpy as np

# dummy radial grid and density values for hydrogen
r = np.linspace(0.001, 10.0, 1000)
dens = np.exp(-2 * r) / np.pi  #H 1s density

raw_file = os.path.join(raw_path, "H_0_2.mydata")
#H_0_2 is just a naming convention H is element symbol 0 is charge and 2 is multiplicity
np.savetxt(raw_file, np.column_stack([r, dens]))
print("Raw file written:", raw_file)

Raw file written: /home/username/.local/lib/python3.13/site-packages/atomdb/datasets/mydata/raw/H_0_2.mydata


In [8]:
#Creating the run.py file

run_py = '''
import os
import numpy as np
import atomdb
from atomdb.periodic import Element

def run(elem, charge, mult, nexc, dataset, datapath):
    # resolve element info
    elem = atomdb.element_symbol(elem)
    atnum = atomdb.element_number(elem)
    nelec = atnum - charge
    nspin = mult - 1

    # load raw data file
    raw_file = os.path.join(datapath, dataset, "raw", f"{elem}_{charge}_{mult}.mydata")
    data = np.loadtxt(raw_file)
    rs, dens_tot = data[:, 0], data[:, 1]

    # build fields and return Species
    fields = dict(
        elem=elem,
        atnum=atnum,
        obasis_name=dataset,
        nelec=nelec,
        nspin=nspin,
        nexc=nexc,
        atmass=Element(elem).mass,
        energy=-0.5,
        rs=rs,
        dens_tot=dens_tot,
    )
    return atomdb.Species(dataset, fields)
'''

run_file = os.path.join(dataset_path, "run.py")
with open(run_file, "w") as f:
    f.write(run_py)

# also need an __init__.py so Python treats it as a package
with open(os.path.join(dataset_path, "__init__.py"), "w") as f:
    f.write("")

print("run.py written:", run_file)

run.py written: /home/username/.local/lib/python3.13/site-packages/atomdb/datasets/mydata/run.py


In [9]:
# Step 1 — compile() runs your run.py AND dumps to disk in one go
atomdb.compile("H", charge=0, mult=2, nexc=0, dataset="mydata", datapath=datapath)

# Step 2 — load() reads what compile() wrote to disk
sp = atomdb.load("H", charge=0, mult=2, nexc=0, dataset="mydata", datapath=datapath)
print("Element:      ", sp.elem)
print("Atomic number:", sp.atnum)
print("Electrons:    ", sp.nelec)
print("Charge:       ", sp.charge)
print("Mult:         ", sp.mult)
print("Energy:       ", sp.energy)

Element:       H
Atomic number: 1
Electrons:     1
Charge:        0
Mult:          2
Energy:        -0.5
